In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# Step 1: Input Function with Added Features
def collect_user_inputs():
    gender = input("Enter Gender (Male/Female/Others): ").strip()
    age = int(input("Enter your age: "))  # New feature
    activity_level = input("Enter Activity Level (Low/High/Moderate): ").strip()
    calorie_intake = float(input("Enter your daily calorie intake: "))
    water_intake = float(input("Enter your daily water intake (in liters): "))
    sleep_hours = float(input("Enter your daily sleep hours: "))
    height = float(input("Enter your height (in cm): "))
    weight = float(input("Enter your weight (in kg): "))
    body_fat = float(input("Enter your body fat percentage: "))  # New feature
    exercise_frequency = int(input("How many days per week do you exercise? "))  # New feature
    stress_level = input("Enter your stress level (Low/Moderate/High): ").strip()  # New feature
    weather = input("Enter current weather (Sunny/Rainy/Winter): ").strip()
    dietary_preference = input("Enter your meal preference (Vegetarian/Vegan/Non-Vegetarian/Eggitarian): ").strip()  # New feature
    chronic_conditions = input("Do you have any chronic health conditions (Yes/No)? ").strip()  # New feature

    return {
        'Gender': gender,
        'Age': age,
        'Activity_Level': activity_level,
        'Calories_Intake': calorie_intake,
        'Water_Intake': water_intake,
        'Sleep_Hours': sleep_hours,
        'Height_cm': height,
        'Weight_kg': weight,
        'Body_Fat': body_fat,
        'Exercise_Frequency': exercise_frequency,
        'Stress_Level': stress_level,
        'Weather': weather,
        'Dietary_Preference': dietary_preference,
        'Chronic_Conditions': chronic_conditions
    }

# Step 2: BMI Calculation (Already Defined)
def calculate_bmi(height_cm, weight_kg):
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    return bmi

# Step 3: Preprocessing Function with New Features
def preprocess_input(user_input, reference_columns=None):
    user_input['BMI'] = calculate_bmi(user_input['Height_cm'], user_input['Weight_kg'])
    df = pd.DataFrame([user_input])

    # One-hot encode categorical variables
    df = pd.get_dummies(df, columns=['Gender', 'Activity_Level', 'Weather', 'Stress_Level', 'Dietary_Preference', 'Chronic_Conditions'], drop_first=True)

    # Ensure all reference columns exist
    if reference_columns is not None:
        for col in reference_columns:
            if col not in df.columns:
                df[col] = 0
        df = df[reference_columns]  # Ensure same column order
    return df

# Step 4: Train Dummy Model with New Features
def train_model():
    np.random.seed(42)
    data = pd.DataFrame({
        'Calories_Intake': np.random.randint(1500, 3000, 200),
        'Water_Intake': np.random.uniform(1, 4, 200),
        'Sleep_Hours': np.random.uniform(4, 10, 200),
        'Height_cm': np.random.uniform(150, 190, 200),
        'Weight_kg': np.random.uniform(50, 100, 200),
        'Body_Fat': np.random.uniform(10, 30, 200),  # New feature
        'Exercise_Frequency': np.random.randint(0, 7, 200),  # New feature
        'Stress_Level': np.random.choice(['Low', 'Moderate', 'High'], 200),  # New feature
        'Gender': np.random.choice(['Male', 'Female', 'Others'], 200),
        'Activity_Level': np.random.choice(['Low', 'Moderate', 'High'], 200),
        'Weather': np.random.choice(['Sunny', 'Rainy', 'Winter'], 200),
        'Dietary_Preference': np.random.choice(['Vegetarian', 'Vegan', 'Non-Vegetarian', 'Eggitarian'], 200),  # New feature
        'Chronic_Conditions': np.random.choice(['Yes', 'No'], 200),  # New feature
    })

    data['BMI'] = data.apply(lambda row: calculate_bmi(row['Height_cm'], row['Weight_kg']), axis=1)

    # Dummy target: Health_Score (synthetic for regression)
    data['Health_Score'] = (
        0.3 * data['Calories_Intake'] +
        5 * data['Water_Intake'] +
        10 * data['Sleep_Hours'] -
        2 * data['BMI'] +
        0.5 * data['Body_Fat'] +  # Include body fat influence
        3 * data['Exercise_Frequency'] -  # Include exercise frequency influence
        1 * (data['Stress_Level'] == 'High') +  # Adjust stress level influence
        np.random.normal(0, 10, 200)  # Add noise
    )

    X = data.drop('Health_Score', axis=1)
    y = data['Health_Score']

    # One-hot encoding
    X = pd.get_dummies(X, columns=['Gender', 'Activity_Level', 'Weather', 'Stress_Level', 'Dietary_Preference', 'Chronic_Conditions'], drop_first=True)

    # Save reference columns for future input processing
    reference_columns = X.columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)

    xgb_model = XGBRegressor(n_estimators=100, random_state=42)
    xgb_model.fit(X_train, y_train)

    # Evaluate models
    rf_preds = rf_model.predict(X_test)
    xgb_preds = xgb_model.predict(X_test)

    print("Random Forest MSE:", mean_squared_error(y_test, rf_preds))
    print("XGBoost MSE:", mean_squared_error(y_test, xgb_preds))

    return rf_model, reference_columns

# Step 5: Generate Dietary Plan using LLM (Example)
def generate_dietary_plan(user_data):
    # For simplicity, a mock dietary plan is generated.
    # In a real-world scenario, integrate with an LLM (like GPT-4 or OpenAI API).
    dietary_plan = f"Based on your preferences (Dietary Preference: {user_data['Dietary_Preference']}, Stress Level: {user_data['Stress_Level']}, Body Fat: {user_data['Body_Fat']}%), here is your tailored dietary plan: \n\n" \
                   "- Breakfast: Protein-packed smoothie with almond milk and chia seeds\n" \
                   "- Lunch: Grilled chicken salad with quinoa\n" \
                   "- Dinner: Stir-fried vegetables with tofu or a lean meat of your choice\n" \
                   "- Snacks: Nuts, fruits, and a low-sugar yogurt."
    return dietary_plan

def modify_dietary_plan(user_data):
    # Modification function to update dietary plans based on user's feedback
    new_plan = f"Based on your feedback and changes, your updated dietary plan is:\n\n" \
               "- Breakfast: Oats with fruits and chia seeds\n" \
               "- Lunch: Grilled fish with avocado and salad\n" \
               "- Dinner: Grilled vegetables with a protein source\n" \
               "- Snacks: Protein bars, almonds, and fruits."
    return new_plan

# Step 6: Main Execution
if __name__ == "__main__":
    model, ref_cols = train_model()
    print("\n--- Enter your personal health data below ---\n")
    user_data = collect_user_inputs()
    user_df = preprocess_input(user_data, ref_cols)
    prediction = model.predict(user_df)[0]
    print(f"\nPredicted Health Score: {prediction:.2f}")

    # Call LLM for dietary plan recommendation (assuming OpenAI integration here)
    dietary_plan = generate_dietary_plan(user_data)
    print(f"\nTailored Dietary Plan: {dietary_plan}")

    # Ask user if changes are needed
    changes_needed = input("Would you like to make any changes to your dietary plan (Yes/No)? ").strip()
    if changes_needed.lower() == "yes":
        dietary_plan = modify_dietary_plan(user_data)
        print(f"\nUpdated Dietary Plan: {dietary_plan}")
